In [40]:
import os

from mistralai.client import Mistral
from pathlib import Path
from pprint import pprint

# https://console.mistral.ai/build/playground?agentId=ag_01a0101448cc76ad85385f677839f8b4&from=agents

client = Mistral(api_key=os.environ.get("MISTRAL_API_KEY"))

def send_prompt(prompt:str):
    inputs = [
        {
            "role":"user",
            "content":prompt
        }
    ]

    return client.beta.conversations.start(
        agent_id="ag_01a0101448cc76ad85385f677839f8b4",
        #agent_version=8, # utiliser la dernière version
        inputs=inputs,
    )

def get_response(response:any):
    for output in response.outputs:
        if output.type == "message.output":
            return output.content
    return None

            
def send_and_print(prompt:str):
    print(get_response(send_prompt(prompt)))

In [41]:
send_and_print("quel sont les horiaires d'ouvertures ?")

La mairie de Trifouillis-sur-Loire est ouverte du lundi au vendredi de 9h00 à 17h30 et le samedi de 9h00 à 12h30. Elle est fermée le dimanche.


# Evalue la pertinance du RAG avec RAGA

In [42]:
import faiss
from sentence_bert import make_embeddings as embeddings_SentenceBERT
from mistral import make_embeddings as embeddings_Mistral
from fast_text import make_embeddings as embeddings_FastText
import pickle
import numpy as np

# charge la base de données et les metadatas
with open("faiss_index.meta", "rb") as f:
    metadata = pickle.load(f)

index = faiss.read_index("faiss_index.idx")

# index des modeles
make_embeddings_by_index = [
    embeddings_SentenceBERT,
    embeddings_Mistral,
    embeddings_FastText
]

k = 5 # Nombre de documents extraits pour construire le contexte RAG

model_index = 0

questions_test = [
    "Quel est le nom du Maire ?",
    "Quels ont été les principaux projets en 2023 ?",
    "Quelles sont les horaires d'ouverture de la mairie ?",
    "Résume moi le règlement municipal ",
    "Combien de km de pistes cyclables allons nous disposer ?",
    "Bonjour, quel temps fait-il aujourd'hui ?"
]

answers = []

placeholder_contexts = []

ground_truths = [
    "Le nom du Maire est Madame Pétillante Rigolade.",
    "Les principaux projets en 2023 concernent la voirie, l'éclairage public, les espaces verts, l'école primaire, un pôle sportif, la place du marché, la surveillance urbaine, les pistes cyclables et le tri sélectif.",
    "La mairie est ouverte du lundi au vendredi de 8h30 à 12h00.",
    "Le règlement municipal vise principalement à garantir l'ordre public, la sécurité, la tranquillité et la salubrité dans la commune pour tous.",
    "Le plan d'action 2026 prévoit environ 2,5 km de nouvelles pistes cyclables sécurisées.",
    "Le système ne peut pas donner la météo en temps réel et suggère de consulter une source externe." # Même pour une réponse "je ne sais pas", la ground_truth est utile
]

for query in questions_test:
    # Génération de l'embedding de la requête
    query_embedding = make_embeddings_by_index[model_index](query)

    # Recherche des k documents les plus similaires
    distances, indices = index.search(np.array([query_embedding]), k)

    # Construit un prompt unique à partir des résultats

    chunks = [
        metadata[index]["text"]
        for index in indices[0]
    ]

    context = "\n\n".join(chunks)

    prompt = f"""
        Contexte :
        {context}

        Question :
        {query}
        """

    response = send_prompt(prompt)

    response_text = get_response(response)

    answers.append(response_text)
    
    placeholder_contexts.append(chunks)

# crée le jeu de données
evaluation_data = {
    # La question posée par l'utilisateur
    "user_input": questions_test,
    # La réponse générée par votre système RAG actuel en réponse à cette question
    "response": answers,
    # La liste des segments de texte (chunks) que votre système RAG a récupérés de la base de connaissances pour générer cette réponseanswer
    "retrieved_contexts": placeholder_contexts,
    # La réponse idéale ou les informations clés que la réponse aurait dû contenir. C'est la "vérité terrain" qui permet une évaluation plus fiable, notamment pour la métriquecontext_recall
    "reference": ground_truths
}

# sauvegarde les données
with open("ragas.dat", "wb") as f:
    pickle.dump(evaluation_data, f)

In [43]:
import pprint
import pickle

# charge les données
if not "evaluation_data" in globals():
    with open("ragas.dat", "rb") as f:
        evaluation_data = pickle.load(f)


In [44]:
from ragas import EvaluationDataset

evaluation_data = [
    {
        "user_input": question,
        "retrieved_contexts": context,
        "response": response,
        "reference": reference,
    }
    for question, context, response, reference
    in zip(
        evaluation_data["user_input"],
        evaluation_data["retrieved_contexts"],
        evaluation_data["response"],
        evaluation_data["reference"]
    )
]

evaluation_dataset = EvaluationDataset.from_list(evaluation_data)

In [45]:
pprint.pp(evaluation_data)

[{'user_input': 'Quel est le nom du Maire ?',
  'retrieved_contexts': ['Le projet se déroulera selon les étapes suivantes :',
                         '- o Une collecte organisée sur un calendrier '
                         'précis.\n'
                         '- o La mise à disposition de bacs adaptés à chaque '
                         'type de déchet.\n'
                         "- o Un suivi rigoureux pour garantir l'hygiène et la "
                         'salubrité publique.',
                         '<!-- image -->\n'
                         '\n'
                         'concitoyens,\n'
                         '\n'
                         'Nous avons le plaisir de vous accueillir sous '
                         "l'égide de la bonne humeur et du sérieux nécessaire "
                         "à l'exercice de nos fonctions. La Mairie de "
                         'Triffouillis sur Loire vous invite à consulter les '
                         'informations ci-après afin de mieu

In [48]:
import os
import asyncio
import numpy as np

from openai import AsyncOpenAI

from ragas.llms import llm_factory
from ragas.embeddings.base import embedding_factory

from ragas.metrics.collections import (
    Faithfulness,
    AnswerRelevancy,
    ContextPrecisionWithReference,
    ContextRecall,
)


# ============================================================
# 1. Client Mistral avec API compatible OpenAI
# ============================================================

mistral_client = AsyncOpenAI(
    api_key=os.environ["MISTRAL_API_KEY"],
    base_url="https://api.mistral.ai/v1",
)


# ============================================================
# 2. LLM utilisé par Ragas
# ============================================================

evaluator_llm = llm_factory(
    model="mistral-small-latest",
    provider="openai",
    client=mistral_client,
)


# ============================================================
# 3. Embeddings utilisés par Ragas
# ============================================================

evaluator_embeddings = embedding_factory(
    provider="openai",
    model="mistral-embed",
    client=mistral_client,
)


# ============================================================
# 4. Initialisation des métriques
# ============================================================

metrics = {
    "faithfulness": Faithfulness(
        llm=evaluator_llm,
    ),

    "answer_relevancy": AnswerRelevancy(
        llm=evaluator_llm,
        embeddings=evaluator_embeddings,
    ),

    "context_precision": ContextPrecisionWithReference(
        llm=evaluator_llm,
    ),

    "context_recall": ContextRecall(
        llm=evaluator_llm,
    ),
}


# ============================================================
# 5. Fonction utilitaire
# ============================================================

def get_score(result):

    if result is None:
        return np.nan

    value = result.value

    if value is None:
        return np.nan

    try:
        value = float(value)
    except (TypeError, ValueError):
        return np.nan

    if np.isnan(value):
        return np.nan

    return value


# ============================================================
# 6. Évaluation
# ============================================================

results = []

for i, sample in enumerate(evaluation_dataset):

    print()
    print("=" * 80)
    print(f"Évaluation {i + 1}/{len(evaluation_dataset)}")
    print("=" * 80)
    print(f"Question : {sample.user_input}")
    print(f"Réponse : {sample.response}")

    result = {
        "user_input": sample.user_input,
        "faithfulness": np.nan,
        "answer_relevancy": np.nan,
        "context_precision": np.nan,
        "context_recall": np.nan,
    }


    # --------------------------------------------------------
    # Faithfulness
    # --------------------------------------------------------

    try:

        score = await metrics["faithfulness"].ascore(
            user_input=sample.user_input,
            response=sample.response,
            retrieved_contexts=sample.retrieved_contexts,
        )

        result["faithfulness"] = get_score(score)

        print(
            f"Faithfulness      : "
            f"{result['faithfulness']:.4f}"
        )

    except Exception as e:

        print(
            f"Faithfulness      : ERREUR - "
            f"{type(e).__name__}: {e}"
        )

    await asyncio.sleep(2)


    # --------------------------------------------------------
    # Answer Relevancy
    # --------------------------------------------------------

    try:

        score = await metrics["answer_relevancy"].ascore(
            user_input=sample.user_input,
            response=sample.response,
        )

        result["answer_relevancy"] = get_score(score)

        print(
            f"Answer Relevancy  : "
            f"{result['answer_relevancy']:.4f}"
        )

    except Exception as e:

        print(
            f"Answer Relevancy  : ERREUR - "
            f"{type(e).__name__}: {e}"
        )

    await asyncio.sleep(2)


    # --------------------------------------------------------
    # Context Precision
    # --------------------------------------------------------

    try:

        score = await metrics["context_precision"].ascore(
            user_input=sample.user_input,
            retrieved_contexts=sample.retrieved_contexts,
            reference=sample.reference,
        )

        result["context_precision"] = get_score(score)

        print(
            f"Context Precision : "
            f"{result['context_precision']:.4f}"
        )

    except Exception as e:

        print(
            f"Context Precision : ERREUR - "
            f"{type(e).__name__}: {e}"
        )

    await asyncio.sleep(2)


    # --------------------------------------------------------
    # Context Recall
    # --------------------------------------------------------

    try:

        score = await metrics["context_recall"].ascore(
            user_input=sample.user_input,
            retrieved_contexts=sample.retrieved_contexts,
            reference=sample.reference,
        )

        result["context_recall"] = get_score(score)

        print(
            f"Context Recall    : "
            f"{result['context_recall']:.4f}"
        )

    except Exception as e:

        print(
            f"Context Recall    : ERREUR - "
            f"{type(e).__name__}: {e}"
        )


    # --------------------------------------------------------
    # Ajout du résultat
    # --------------------------------------------------------

    results.append(result)

    # Pause avant l'exemple suivant
    await asyncio.sleep(3)


# ============================================================
# 7. Résultats détaillés
# ============================================================

print()
print()
print("=" * 100)
print("RÉSULTATS DÉTAILLÉS")
print("=" * 100)

print(
    f"{'Question':<55}"
    f"{'Faith.':>10}"
    f"{'Relev.':>10}"
    f"{'Prec.':>10}"
    f"{'Recall':>10}"
)

print("-" * 100)

for result in results:

    question = result["user_input"]

    # Réduction de la question pour l'affichage
    if len(question) > 52:
        question = question[:49] + "..."

    faithfulness = result["faithfulness"]
    answer_relevancy = result["answer_relevancy"]
    context_precision = result["context_precision"]
    context_recall = result["context_recall"]

    def format_score(value):
        if np.isnan(value):
            return "NaN"
        return f"{value:.4f}"

    print(
        f"{question:<55}"
        f"{format_score(faithfulness):>10}"
        f"{format_score(answer_relevancy):>10}"
        f"{format_score(context_precision):>10}"
        f"{format_score(context_recall):>10}"
    )


# ============================================================
# 8. Calcul des moyennes
# ============================================================

print()
print("=" * 60)
print("MOYENNES")
print("=" * 60)


metric_names = [
    "faithfulness",
    "answer_relevancy",
    "context_precision",
    "context_recall",
]


averages = {}

for metric_name in metric_names:

    values = [
        result[metric_name]
        for result in results
        if not np.isnan(result[metric_name])
    ]

    if values:

        averages[metric_name] = np.mean(values)

        print(
            f"{metric_name:<20} : "
            f"{averages[metric_name]:.4f}"
        )

    else:

        averages[metric_name] = np.nan

        print(
            f"{metric_name:<20} : NaN"
        )


# ============================================================
# 9. Score global
# ============================================================

valid_averages = [
    value
    for value in averages.values()
    if not np.isnan(value)
]

if valid_averages:

    global_score = np.mean(valid_averages)

    print()
    print("=" * 60)
    print(f"SCORE GLOBAL         : {global_score:.4f}")
    print("=" * 60)

else:

    print()
    print("Impossible de calculer le score global.")


Évaluation 1/6
Question : Quel est le nom du Maire ?
Faithfulness      : 1.0000
Answer Relevancy  : 0.8743
Context Precision : 0.3333
Context Recall    : 1.0000

Évaluation 2/6
Question : Quels ont été les principaux projets en 2023 ?
Faithfulness      : 0.8000
Answer Relevancy  : 0.8559
Context Precision : 0.0000
Context Recall    : 0.1111

Évaluation 3/6
Question : Quelles sont les horaires d'ouverture de la mairie ?
Faithfulness      : 0.3333
Answer Relevancy  : 0.9412
Context Precision : 0.0000
Context Recall    : 0.0000

Évaluation 4/6
Question : Résume moi le règlement municipal 
Faithfulness      : 0.0000
Answer Relevancy  : 0.0000
Context Precision : 0.0000
Context Recall    : 0.0000

Évaluation 5/6
Question : Combien de km de pistes cyclables allons nous disposer ?
Faithfulness      : 0.3333
Answer Relevancy  : 0.0000
Context Precision : 0.8667
Context Recall    : 0.0000

Évaluation 6/6
Question : Bonjour, quel temps fait-il aujourd'hui ?
Faithfulness      : 0.0000
Answer Rel